# Multimodal RAG — CLIP + BLIP + Corrective RAG

A thin Colab notebook. All the logic lives in the `multimodal_rag/` package on GitHub — this notebook just clones it, installs dependencies, and calls it.

**Before running:** make sure `REPO_URL` below points at your own repo (see `README.md` in the package for setup details).

In [ ]:
# Optional: run this only if you need a completely fresh clone
# (e.g. after switching repos, or to rule out a stale local copy while debugging).
!rm -rf /content/multimodal_rag_repo


In [ ]:
REPO_URL = "https://github.com/BerehanAbulezz/multimodal_rag_.git"
REPO_DIR = "multimodal_rag_"


## 1) Clone the repo and install dependencies

Clones (or pulls, if already present) the package repo, `cd`s into it, installs everything in `requirements.txt`, and puts the repo on `sys.path` so `import multimodal_rag` works.

In [ ]:
import os

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull


In [ ]:
%cd {REPO_DIR}
!pip install -q -r requirements.txt


In [ ]:
import sys
sys.path.insert(0, os.getcwd())


## 2) Set up the Gemini API key and load the package

Reads `GOOGLE_API_KEY` from Colab secrets (Settings → Secrets) and initializes the Gemini client used for answer generation, grading, and query rewriting.

In [ ]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

import multimodal_rag as mrag
mrag.init_client(GOOGLE_API_KEY)


> The first call to `mrag.ask(...)` builds the multimodal index (CLIP embeddings for all 50 text entries + 12 images in `data/`). This only happens once per session — it's cached in memory afterwards.

## 3) Text-only query

A plain text question. Retrieval searches the shared CLIP text/image space, Gemini generates an answer from the retrieved context, and the Corrective RAG loop grades the answer and retries with a rewritten query if it's ungrounded or incomplete.

In [ ]:
result = mrag.ask("What do giraffes eat and how does their long neck help them?")

print("Answer:\n", result["answer"])
print("\nAttempts:", result["attempts"])
if "warning" in result:
    print("WARNING:", result["warning"])

print("\n--- Retrieved items ---")
for item, score in result["results"]:
    print(f"[{item.type}] score={score:.3f} -> {item.source.get('animal')}: "
          f"{item.content if item.type == 'text' else item.content}")


## 4) Image query (bundled sample image)

Uses one of the animal photos already included in `data/images/` as the query. Under the hood this runs a hybrid search: a BLIP caption of the image is used for a text-mode search (to pull in relevant facts), combined with a direct image-mode search (to pull in the closest matching photo).

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

image_path = "data/images/elephant1.jpg"  # try any file under data/images/

plt.imshow(Image.open(image_path))
plt.axis("off")
plt.show()

result = mrag.ask(
    "What animal is this and what is interesting about it?",
    image_path=image_path,
)

print("Answer:\n", result["answer"])
print("\nAttempts:", result["attempts"])


## 4b) Optional: upload and ask about your own image

In [ ]:
from google.colab import files

uploaded_image = files.upload()
my_image_path = list(uploaded_image.keys())[0]

plt.imshow(Image.open(my_image_path))
plt.axis("off")
plt.show()

result = mrag.ask("What animal is this and what is interesting about it?", image_path=my_image_path)
print("Answer:\n", result["answer"])


## 5) PDF ingestion

Upload any PDF. It gets text-extracted, chunked, embedded with the CLIP text encoder, and appended to the live index — so its content becomes searchable alongside the animal dataset for the rest of this session.

In [ ]:
from google.colab import files

uploaded_pdf = files.upload()
pdf_path = list(uploaded_pdf.keys())[0]
print("Using PDF:", pdf_path)


### 5a) Summarize the uploaded PDF

This first call passes `pdf_path`, so it ingests the PDF into the index before answering. Later calls in this section omit `pdf_path` since the content is already indexed.

In [ ]:
result = mrag.ask(
    "Summarize the main idea of the uploaded document in a few sentences.",
    pdf_path=pdf_path,
)
print("Answer:\n", result["answer"])
print("Attempts:", result["attempts"])


### 5b) Ask a specific question about the PDF content

No `pdf_path` needed here — the document is already part of the index from the previous cell.

In [ ]:
result = mrag.ask("What is one specific fact mentioned in the uploaded document?")
print("Answer:\n", result["answer"])
print("Attempts:", result["attempts"])


### 5c) Compare the PDF content with the existing animal dataset

A cross-source question: the retriever pulls context from BOTH the PDF chunks and the original animal dataset (text + images), and Gemini is asked to compare them. Edit the question below to match whatever your PDF is actually about.

In [ ]:
result = mrag.ask(
    "Compare what the uploaded document says with the information about "
    "animals in the existing dataset. What is similar, and what is different?"
)
print("Answer:\n", result["answer"])
print("Attempts:", result["attempts"])


## 6) Inspect the Corrective RAG trace

Every attempt (grading verdict, search query used, and the generated answer at that step) for the most recent `ask()` call is recorded in `result["trace"]`.

In [ ]:
for step in result["trace"]:
    print(f"Attempt {step['attempt']} | correct={step['graded_correct']}")
    print("query:", step["search_query"])
    print("answer:", step["answer"][:200], "...")
    print("---")
